In [1]:
# =========================================================
# 1️⃣ Mount Google Drive
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# 2️⃣ Import Libraries
# =========================================================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# 3️⃣ Base Path (তোমার Drive location)
# =========================================================
BASE = "/content/drive/MyDrive/Thesis file/Feature_Extract_Formants"

# =========================================================
# 4️⃣ Feature Extractors + Classifiers
# =========================================================
extractors = [
    'AlexNet', 'CustomCNN', 'DenseNet121', 'EfficientNetB0',
    'InceptionV3', 'MobileNetV2', 'ResNet50', 'VGG16', 'ViT', 'Xception'
]

classifiers = {
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42)
}

# =========================================================
# 5️⃣ Results DataFrame
# =========================================================
results = pd.DataFrame(columns=['Feature Extractor'] + list(classifiers.keys()) + ['Best', 'Best Accuracy'])
results['Feature Extractor'] = extractors

# =========================================================
# 6️⃣ Loop over each feature extractor
# =========================================================
for extractor in extractors:
    print(f"\n{'='*60}")
    print(f"Processing {extractor} → F2 + F3 + F4")

    # ---------------- FILES ----------------
    f1_file = f"{BASE}/F1_Features_{extractor}.csv"   # labels here
    f2_file = f"{BASE}/F2_Features_{extractor}.csv"
    f3_file = f"{BASE}/F3_Features_{extractor}.csv"
    f4_file = f"{BASE}/F4_Features_{extractor}.csv"

    # Load data
    try:
        df1 = pd.read_csv(f1_file)
        df2 = pd.read_csv(f2_file)
        df3 = pd.read_csv(f3_file)
        df4 = pd.read_csv(f4_file)
    except FileNotFoundError:
        print(f"❌ Files not found for {extractor}. Skipping...")
        for name in classifiers.keys():
            results.loc[results['Feature Extractor']==extractor, name] = "NA"
        results.loc[results['Feature Extractor']==extractor, ['Best','Best Accuracy']] = "NA","NA"
        continue

    # ---------------- Label Column from F1 ----------------
    possible_labels = ['label', 'class', 'target', 'phoneme', 'category', 'Label', 'Class', 'phoneme_label']
    label_col = None
    for col in possible_labels:
        if col in df1.columns:
            label_col = col
            break
    if label_col is None:
        label_col = df1.columns[1]  # assume 2nd column if not found
        print(f"   → Label assumed from column: {label_col}")

    y = df1[label_col]

    # ---------------- Clean function ----------------
    def clean(df):
        df = df.copy()
        drop_cols = [c for c in df.columns if any(x in c.lower() for x in ['filename','path','unnamed','index'])]
        df.drop(columns=drop_cols, errors='ignore', inplace=True)
        return df

    df2 = clean(df2)
    df3 = clean(df3)
    df4 = clean(df4)

    # drop label in case exists
    for d in [df2, df3, df4]:
        if label_col in d.columns:
            d.drop(columns=[label_col], inplace=True)

    # ---------------- Merge Features ----------------
    X = pd.concat([df2.reset_index(drop=True),
                   df3.reset_index(drop=True),
                   df4.reset_index(drop=True)], axis=1)

    print(f"   → Final feature shape: {X.shape} | Classes: {y.nunique()}")

    # ---------------- Train/Test Split ----------------
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                            test_size=0.2,
                                                            random_state=42,
                                                            stratify=y)
    except:
        X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                            test_size=0.2,
                                                            random_state=42)
        print("   → Stratify failed, normal split used")

    # ---------------- Run Classifiers ----------------
    best_acc = 0
    best_name = ''

    for name, clf in classifiers.items():
        try:
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            acc = round(accuracy_score(y_test, y_pred),4)
            results.loc[results['Feature Extractor']==extractor, name] = acc
            if acc > best_acc:
                best_acc = acc
                best_name = name
            print(f"   → {name.ljust(20)} : {acc}")
        except Exception as e:
            results.loc[results['Feature Extractor']==extractor, name] = 0.0
            print(f"   → {name} failed: {e}")

    results.loc[results['Feature Extractor']==extractor,'Best'] = best_name
    results.loc[results['Feature Extractor']==extractor,'Best Accuracy'] = best_acc
    print(f"   → BEST MODEL = {best_name} ({best_acc})")

# =========================================================
# 7️⃣ Save final Excel
# =========================================================
output_file = f"{BASE}/Summary_F2_F3_F4.xlsx"
results.to_excel(output_file, index=False)

print("\n============================================================")
print("🎉 ALL DONE! Summary saved to:")
print(output_file)

# Display final results
results


Mounted at /content/drive

Processing AlexNet → F2 + F3 + F4
   → Final feature shape: (523, 27648) | Classes: 2
   → AdaBoost             : 0.8762
   → Decision Tree        : 0.819
   → Extra Trees          : 0.8952
   → Gradient Boosting    : 0.9048
   → KNN                  : 0.9333
   → Logistic Regression  : 0.9619
   → Naive Bayes          : 0.8
   → Random Forest        : 0.9048
   → SVM                  : 0.9524
   → BEST MODEL = Logistic Regression (0.9619)

Processing CustomCNN → F2 + F3 + F4
   → Final feature shape: (523, 301056) | Classes: 2
   → AdaBoost             : 0.9048
   → Decision Tree        : 0.8095
   → Extra Trees          : 0.8952
   → Gradient Boosting    : 0.9238
   → KNN                  : 0.8
   → Logistic Regression  : 0.9429
   → Naive Bayes          : 0.6667
   → Random Forest        : 0.8952
   → SVM                  : 0.9429
   → BEST MODEL = Logistic Regression (0.9429)

Processing DenseNet121 → F2 + F3 + F4
   → Final feature shape: (523, 3072) | C

,Feature Extractor,AdaBoost,Decision Tree,Extra Trees,Gradient Boosting,KNN,Logistic Regression,Naive Bayes,Random Forest,SVM,Best,Best Accuracy
0,AlexNet,0.8762,0.819,0.8952,0.9048,0.9333,0.9619,0.8,0.9048,0.9524,Logistic Regression,0.9619
1,CustomCNN,0.9048,0.8095,0.8952,0.9238,0.8,0.9429,0.6667,0.8952,0.9429,Logistic Regression,0.9429
2,DenseNet121,0.8952,0.8286,0.8857,0.8857,0.8667,0.9429,0.9048,0.8571,0.9238,Logistic Regression,0.9429
3,EfficientNetB0,0.9143,0.7905,0.8762,0.8762,0.8571,0.9143,0.8857,0.8762,0.9048,AdaBoost,0.9143
4,InceptionV3,0.8857,0.7714,0.8571,0.9048,0.819,0.9048,0.9048,0.8476,0.9143,SVM,0.9143
5,MobileNetV2,0.9048,0.819,0.8952,0.8762,0.8857,0.9238,0.8762,0.8667,0.9429,SVM,0.9429
6,ResNet50,0.9238,0.7524,0.8857,0.9143,0.8762,0.9143,0.8762,0.9143,0.9048,AdaBoost,0.9238
7,VGG16,0.9143,0.8,0.9048,0.9238,0.9238,0.9333,0.7524,0.8952,0.9619,SVM,0.9619
8,ViT,0.819,0.7238,0.7905,0.819,0.8095,0.8667,0.819,0.8095,0.8,Logistic Regression,0.8667
9,Xception,0.8952,0.819,0.8381,0.8476,0.8,0.8952,0.819,0.8667,0.8571,AdaBoost,0.8952
